In [43]:
########################################################################################################################################################################
########################################################################################################################################################################
############################################################# PIPELINE BATCH PARA EXTRACCIÓN ESTRUCTURADA ##############################################################
########################################################################################################################################################################
########################################################################################################################################################################

In [85]:
# 1. Librerías.
from __future__ import annotations
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
import pandas as pd
import json, pathlib
from openai import OpenAI
from pprint import pprint
from datetime import datetime

pd.options.display.max_columns = None

In [45]:
#2. Constantes.
nombre_prueba = input("Por favor, asigne un subfijo para el nombre de los archivos output siguiendo el patrón: 'IL1610_1' (inicial nombre, inicial apellido,dia, mes,número de prueba/detalle de prueba):")
project_path = "C:/Users/i_link/Maestría/Text Mining/nlp_dmuba/"
dataset_file_path = project_path + "1-Scraping/dataset_consolidado/df.parquet"
batch_requests_path = project_path + "5-LLMs/openai/pruebas_batch/batch_requests_{}.jsonl".format(nombre_prueba)
batch_results_path =  project_path + "5-LLMs/openai/pruebas_batch/batch_results_{}.jsonl".format(nombre_prueba)
batch_errors_path =   project_path + "5-LLMs/openai/pruebas_batch/batch_errors_{}.jsonl".format(nombre_prueba)
df_final_path = project_path + "5-LLMs/openai/pruebas_batch/df_final_{}.csv".format(nombre_prueba)

In [129]:
#3. Lecturas.
#a. Datos.
df = pd.read_parquet(dataset_file_path)
#b. Clave API.
with open(project_path + "secrets/opeinai_api_key.txt", "r") as f:
    key = f.read().strip()

In [130]:
#4. Genero un Cliente de OpenAI.
client = OpenAI(api_key=key)

In [146]:
########################################### SAMPLEO PARA PRUEBAS ####################################################
sample = 1500
df_sample = df.dropna(subset=["contenido"]).sample(sample, random_state=42).reset_index(drop=True)

In [ ]:
#5. Prompt. 
#a. System y User Prompt.
SYSTEM_PROMPT = '''
Eres un analista económico-financiero especializado en Argentina.
Tu tarea es EXTRAER DATOS ESTRUCTURADOS de una noticia para modelar el MERVAL.

### Requisitos duros (STRICT)
- Devuelve **SOLO** un **JSON plano** (un único objeto) con **todas** las claves del esquema, al mismo nivel.
- **No** anides objetos, **no** agregues texto, ni comentarios, ni explicaciones.
- **No** inventes datos: usa NULL cuando no haya evidencia explícita.
- Tipos estrictos:
  - boolean: true/false en minúsculas.
  - float: usa punto decimal (“0.0”). Rango señales: [-1.0, 1.0].
  - string: literal; si no hay, usa el default indicado.
  - array JSON: nunca como string; sin duplicados; puede ser [].
- **No infieras** tickers si no aparecen de forma literal o inequívoca.
- Mantén coherencia entre “shock” y las señales de mercado.

### Campos y definiciones (con defaults)

- tipo_actor_principal (string): **Entidad predominante** a la que refiere la noticia.  
  Valores: {"gobierno_nacional","bcra","provincia","municipio","empresa_local","empresa_extranjera","sindicato","poder_judicial","congreso","organismo_internacional","desconocido"}.  
  Default: "desconocido".

- nombre_actor_principal (string): **Nombre propio** del actor principal si es claro (p. ej., “banco central (bcra)”, “javier milei”, “ypf”).  
  Default: "desconocido".

- empresas_mencionadas (array[string]): **Nombres legales u organismos** citados (no tickers). Ej: ["ypf","banco nacion","indec"].  
  Deduplicar.  
  Default: [].

- tickers_mencionados (array[string]): **Tickers literales** (BYMA/ADRs) en **MAYÚSCULAS**. Ej: ["GGAL","BMA","YPFD","ALUA","TGSU2"].  
  Incluir solo si aparece el ticker explícitamente. Sin duplicados.  
  Default: [].

- sectores_mencionados (array[string]): **Sectores/industrias** mapeados a la siguiente **TAXONOMÍA cerrada** (elige solo de la lista):  
  ["Finanzas","Cripto / Fintech","Tecnología","Energía","Minería","Educación","Salud","Comercio / Consumo","Transporte / Logística","Gobierno / Sector Público","Política","Justicia","Seguridad","Turismo","Industria / Manufactura","Agropecuario","Construcción / Inmobiliario","Medio Ambiente","Empleo / Laboral","Servicios","Infraestructura","Otros"]  
  Default: [].

- tipo_evento (string): **Categoría del hecho principal**.  
  Valores: {"monetario","fiscal","regulatorio","corporativo","externo","sindical_social","judicial","electoral","otro","desconocido"}.  
  Default: "desconocido".

- shock (string): **Signo cualitativo** del impacto sobre mercados/economía.  
  Valores: {"positivo","negativo","mixto","neutro","desconocido"}.  
  Default: "desconocido".

- caracter (string): **Temporalidad** del evento.  
  Valores: {"retroactivo","vigente","prospectivo","desconocido"}.  
  Default: "desconocido".

- horizonte_dias (integer|null): **Días hasta el impacto** si el texto lo indica (convierte semanas/meses a días).  
  Si no hay mención explícita → null.  
  Default: null.

- merval (float): **Sesgo esperado** para el índice MERVAL (−1: baja fuerte, 0: neutro, +1: sube fuerte).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- volatilidad_merval (float): **Volatilidad esperada** del MERVAL (−1: baja, +1: alta).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- fx_usdars (float): **Sesgo del tipo de cambio USD/ARS** (−1: aprecia ARS, +1: deprecia ARS).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- spread_usd (float): **Brecha dólar oficial vs paralelo** (−1: se estrecha, +1: se amplía).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- tasa_bcra (float): **Sesgo de tasa de política del BCRA** (−1: baja, +1: sube).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- bonos_soberanos (float): **Sesgo del precio** de bonos soberanos (−1: bajan, +1: suben).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- spread_bonos (float): **Spreads de bonos soberanos** (−1: se estrechan, +1: se amplían).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- actividad_economica (float): **Nivel de actividad** (−1: baja, +1: sube).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- volumen_mercado (float): **Actividad de trading** (−1: bajo, +1: alto).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- valencia_general (float): **Tono general** del artículo sobre economía/mercados (−1: negativo, +1: positivo).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- gobernanza (float): **Tono respecto a gobierno/instituciones** (−1: negativo, +1: positivo).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- expectativa_macro_corto (float): **Expectativa macro** a 1–3 meses (−1: pesimista, +1: optimista).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- expectativa_macro_largo (float): **Expectativa macro** a >6 meses (−1: pesimista, +1: optimista).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- expectativa_fin_corto (float): **Expectativa financiera** a 1–3 meses (−1: negativo, +1: positivo).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- expectativa_fin_largo (float): **Expectativa financiera** a >6 meses (−1: negativo, +1: positivo).  
  Rango: [-1.0, 1.0]. Default: 0.0.

- menciona_inflacion (boolean): True si el texto menciona **inflación/precios**. Default: false.
- menciona_pbi (boolean): True si menciona **PBI/crecimiento**. Default: false.
- menciona_reservas (boolean): True si menciona **reservas del BCRA**. Default: false.
- menciona_embi (boolean): True si menciona **riesgo país/EMBI**. Default: false.
- menciona_deuda (boolean): True si menciona **deuda pública/privada**. Default: false.
- menciona_fmi (boolean): True si menciona **FMI** o acuerdos. Default: false.
- menciona_salarios_paritarias (boolean): True si menciona **salarios/paritarias**. Default: false.
- menciona_tipo_cambio (boolean): True si menciona **tipo de cambio/dólar**. Default: false.
- menciona_confianza_consumidor (boolean): True si menciona **índice/sentimiento de confianza del consumidor**. Default: false.

- menciona_sector_bancario (boolean): True si menciona **bancos/sistema financiero**. Default: false.
- impacto_sector_bancario (float): **Sesgo sobre bancos** (−1: negativo, +1: positivo). Default: 0.0.

- menciona_sector_energia (boolean): True si menciona **energía** (YPF, Pampa, gas, electricidad, petróleo). Default: false.
- impacto_sector_energia (float): **Sesgo sobre energía** (−1: negativo, +1: positivo). Default: 0.0.

- menciona_sector_agroexportador (boolean): True si menciona **agro/exportaciones** (soja, maíz, trigo, etc.). Default: false.
- impacto_sector_agroexportador (float): **Sesgo sobre agro** (−1: negativo, +1: positivo). Default: 0.0.

- menciona_sector_industrial (boolean): True si menciona **industria/manufactura**. Default: false.
- impacto_sector_industrial (float): **Sesgo sobre industria** (−1: negativo, +1: positivo). Default: 0.0.

- menciona_eeuu (boolean): True si menciona **EE.UU.** (economía o mercados). Default: false.
- impacto_eeuu (float): **Impacto esperado de EE.UU.** sobre AR (−1: desfavorable, +1: favorable). Default: 0.0.

- menciona_brasil (boolean): True si menciona **Brasil**. Default: false.
- impacto_brasil (float): **Impacto esperado de Brasil** (−1: desfavorable, +1: favorable). Default: 0.0.

- menciona_mercosur (boolean): True si menciona **Mercosur** o acuerdos regionales. Default: false.
- impacto_mercosur (float): **Impacto esperado del Mercosur** (−1: desfavorable, +1: favorable). Default: 0.0.

- menciona_prestamos_internacionales (boolean): True si menciona **préstamos/financiamiento** de organismos internacionales. Default: false.
- impacto_prestamos_internacionales (float): **Sesgo esperado de dicho financiamiento** (−1: negativo, +1: positivo). Default: 0.0.

- impacto_fmi (float): **Sesgo del impacto del FMI** sobre AR (−1: negativo, +1: positivo). Default: 0.0.

- menciona_commodities (boolean): True si menciona **commodities** (soja, petróleo, oro, litio, etc.). Default: false.
- impacto_commodities (float): **Impacto esperado de commodities** (−1: desfavorable, +1: favorable). Default: 0.0.

- menciona_bolsa_extranjera (boolean): True si menciona **bolsas/índices extranjeros** (S&P 500, Nasdaq, Bovespa, etc.). Default: false.
- impacto_bolsa_extranjera (float): **Impacto esperado de esos mercados** sobre el Merval/AR (−1: desfavorable, +1: favorable). Default: 0.0.

- categoria_fuente (string): **Tipo de fuente** del contenido.  
  Valores: {"oficial","periodistica","analisis","rumor","desconocido"}.  
  Default: "desconocido".

- score_fuente (float): **Confiabilidad** de la fuente según categoría [0..1].  
  Heurística: oficial→0.9, analisis→0.7, periodistica→0.6, rumor→0.2 (usa solo si el texto lo permite).  
  Default: 0.5.

- confianza (float): **Confianza global** de extracción [0..1] (claridad y evidencia).  
  Subir si hay cifras/documentos/citas oficiales/datos verificables; bajar si es vago/opinión/no económico.  
  Default: 0.0.

### Reglas de extracción
1) Usa SOLO el texto provisto; no uses conocimiento externo.
2) “menciona_*”: true si el término o sinónimo aparece explícitamente en el texto.
3) “horizonte_dias”: si hay plazo (“en 2 semanas”, “en 3 meses”), conviértelo a días (14, 90, etc.). Si no, null.
4) “sectores_mencionados”: mapea a la TAXONOMÍA fija (si no encaja, usa “Otros”).
5) “empresas_mencionadas”: nombres legales u organismos; dedup; sin ticker aquí.
6) “tickers_mencionados”: tokens MAYÚSCULAS (p.ej., “ALUA”, “CRES”, “TGSU2”, “YPFD”, “GGAL”, “BMA”). No inventes.
7) No repitas elementos en arrays.

### Heurísticas de consistencia (soft rules)
- Política monetaria contractiva (suba tasa/absorción): tasa_bcra>0 ⇒ merval≤0; bonos_soberanos≤0; fx_usdars≥0.
- Fiscal expansiva sin fondeo: merval mixto; fx_usdars≥0; spread_usd≥0.
- Acuerdo FMI/desembolso confirmado: bonos_soberanos≥0; spread_bonos≤0; fx_usdars≤0; confianza alta.
- Si la noticia **no es económica**: señales=0.0; booleans=false; tipo_evento="desconocido"; confianza ≤ 0.3.
'''


USER_TEMPLATE = '''
Diario: {diario}
Fecha: {fecha}  # YYYY-MM-DD
Seccion: {seccion}
Titulo: {titulo}
Contenido: {contenido}  # truncado a 8000 caracteres si es muy largo.

Instrucciones:
- Devuelve SOLO un JSON plano con todos los campos del esquema del SYSTEM_PROMPT.
- Sin texto adicional ni explicaciones.
- Respeta los tipos de datos, defaults y rangos.
'''

In [171]:
#6. Creo archivo JSONL para carga batch,
with open(batch_requests_path, "w", encoding="utf-8") as f:
    for i, row in df_sample.iterrows():
        contenido = (row.get("contenido") or "")[:5000]
        prompt = USER_TEMPLATE.format(
            diario=row.get("diario", "desconocido"),
            fecha=str(row.get("fecha", "desconocido")),
            seccion=row.get("seccion", "desconocido"),
            titulo=row.get("titulo", "desconocido"),
            contenido=contenido
        )

        request_dict = {
            "custom_id": f"row_{i}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-5-mini", # No se puede usar temperature. Por default ya es determinístico (0).
                "input": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ]
            }

        }
        f.write(json.dumps(request_dict, ensure_ascii=False) + "\n")

print(f"✅ Archivo JSONL creado en batch_request.")

✅ Archivo JSONL creado en batch_request.


In [172]:
#7. Subo el archivo y cargo el batch.
#a. Subo el archivo JSONL.
file_upload = client.files.create(
    file=open(batch_requests_path, "rb"),
    purpose="batch"
)
print("📁 Archivo subido con ID:", file_upload.id)

#b. Creo el batch job usando ese file_id.
batch_job = client.batches.create(
    input_file_id=file_upload.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("🚀 Batch job creado:", batch_job.id)
print("Status inicial:", batch_job.status)

📁 Archivo subido con ID: file-PNZ2CwHKJSCL3FoqZKZ4i9
🚀 Batch job creado: batch_68fac3e2785c8190ac2a21aec275fde5
Status inicial: validating


In [174]:
#8. Conozco el estado de lo que envié.
#a. Consulto.
job = client.batches.retrieve(batch_job.id)
#b. Printeo.
print("📋 Estado:", job.status)
print("⚙️  Output file:", job.output_file_id)
print("📦 ID:", job.id)
print("🕒 Creado:", job.created_at)
pprint(job.model_dump()) # Muestro todos los detalles en bruto.

📋 Estado: failed
⚙️  Output file: None
📦 ID: batch_68fac3e2785c8190ac2a21aec275fde5
🕒 Creado: 1761264610
{'cancelled_at': None,
 'cancelling_at': None,
 'completed_at': None,
 'completion_window': '24h',
 'created_at': 1761264610,
 'endpoint': '/v1/responses',
 'error_file_id': None,
 'errors': {'data': [{'code': 'token_limit_exceeded',
                      'line': None,
                      'message': 'Enqueued token limit reached for gpt-5-mini '
                                 'in organization '
                                 'org-VBo0VEKTMIQp2AjbfcqM3ryt. Limit: '
                                 '5,000,000 enqueued tokens. Please try again '
                                 'once some in_progress batches have been '
                                 'completed.',
                      'param': None}],
            'object': 'list'},
 'expired_at': None,
 'expires_at': 1761351010,
 'failed_at': 1761264612,
 'finalizing_at': None,
 'id': 'batch_68fac3e2785c8190ac2a21aec275fde5',


In [124]:
#9. Conozco errores.
if job.error_file_id:
    #a. Consulto.
    error_file_id = job.error_file_id

    #b. Descargo el archivo con errores.
    error_content = client.files.content(error_file_id)

    #c. Almaceno.
    with open(batch_errors_path.format(nombre_prueba), "wb") as f:
        f.write(error_content.read())

    # d. Printeo.
    print("✅ Archivo de errores descargado en batch_errors")
else:
    print("ℹ️ No hay archivo de errores para este job (error_file_id es None).")

ℹ️ No hay archivo de errores para este job (error_file_id es None).


In [125]:
#10. Descargo resultados, si está completado.
if job.status == "completed":
    output_file_id = job.output_file_id
    result = client.files.content(output_file_id)
    
    # El contenido es un JSONL (una respuesta por línea)
    with open(batch_results_path, "wb") as f:
        f.write(result.read())

    print("✅ Resultados descargados en batch_results.")
else:
    print("ℹ️ Resultados aún no completos.")

✅ Resultados descargados en batch_results.


In [126]:
#11. Armo el dataframe.
if job.status == "completed":
    #a. "Cargo el JSONL completo de respuestas de la API.
    with open(batch_results_path, "r", encoding="utf-8") as f:
        batch_responses = [json.loads(line) for line in f]

    #b. Extraigo toda la info de cada respuesta.
    all_records = []
    for resp in batch_responses:
        try:
            # Extraigo el JSON generado por el modelo.
            text_json_str = resp["response"]["body"]["output"][1]["content"][0]["text"]
            data = json.loads(text_json_str)
            
            # Agrego el custom_id para poder unirlo con el DataFrame original.
            data["custom_id"] = resp.get("custom_id", None)
            all_records.append(data)

        except Exception as e:
            print(f"❌ Error en registro {resp.get('custom_id', 'desconocido')}: {e}")
            continue

    #c. Creo DataFrame plano con todas las columnas extraídas.
    df_features = pd.json_normalize(all_records)

    #d. Agrego columna custom_id al df original para poder hacer merge.
    df_sample['custom_id'] = [f'row_{i}' for i in range(len(df_sample))]

    #e. Uno el df original con las features extraídas.
    df_final = df_sample.merge(df_features, on='custom_id', how='left')

    #f. Elimino custom_id si ya no sirve.
    df_final.drop(columns=["custom_id"], inplace=True)

    #g. Exporto el resultado.
    df_final.to_csv(df_final_path,index=False)
else:
    print("ℹ️ Resultados aún no completos.")

In [127]:
#12. Visualizo cuanto tardó el proceso.
if job.status == "completed":
    #a. Convertimos timestamps a datetime.
    created = datetime.fromtimestamp(job.created_at)
    completed = datetime.fromtimestamp(job.completed_at)

    #b. Calculamos duración.
    duration = completed - created
    print("⏱ Duración del proceso:", duration)
    print("Duración en segundos:", (completed - created).total_seconds())
    print("Duración en minutos:", (completed - created).total_seconds()/60)
else:
    print("ℹ️ Resultados aún no completos.")

⏱ Duración del proceso: 0:05:53
Duración en segundos: 353.0
Duración en minutos: 5.883333333333334


In [128]:
df_final

,diario,fecha,titulo,contenido,url,seccion,tipo_actor_principal,nombre_actor_principal,empresas_mencionadas,tickers_mencionados,sectores_mencionados,tipo_evento,shock,caracter,horizonte_dias,merval,volatilidad_merval,fx_usdars,spread_usd,tasa_bcra,bonos_soberanos,spread_bonos,actividad_economica,volumen_mercado,valencia_general,gobernanza,expectativa_macro_corto,expectativa_macro_largo,expectativa_fin_corto,expectativa_fin_largo,menciona_inflacion,menciona_pbi,menciona_reservas,menciona_embi,menciona_deuda,menciona_fmi,menciona_salarios_paritarias,menciona_tipo_cambio,menciona_confianza_consumidor,menciona_sector_bancario,impacto_sector_bancario,menciona_sector_energia,impacto_sector_energia,menciona_sector_agroexportador,impacto_sector_agroexportador,menciona_sector_industrial,impacto_sector_industrial,menciona_eeuu,impacto_eeuu,menciona_brasil,impacto_brasil,menciona_mercosur,impacto_mercosur,menciona_prestamos_internacionales,impacto_prestamos_internacionales,impacto_fmi,menciona_commodities,impacto_commodities,menciona_bolsa_extranjera,impacto_bolsa_extranjera,categoria_fuente,score_fuente,confianza
0,Ámbito Financiero,2025-01-15,Euro hoy y Euro blue hoy: a cuánto cerró este ...,El euro hoy -sin impuestos- se ofreció este mi...,https://www.ambito.com/finanzas/euro-hoy-y-eur...,finanzas,bcra,banco central (bcra),"[Banco Central, Bitso, Binance, Gobierno, Salv...",[],"[Finanzas, Gobierno / Sector Público, Cripto /...",monetario,mixto,prospectivo,None,0.0,0.2,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.1,0.1,-0.1,0.1,False,False,False,False,False,False,False,True,False,False,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False,0.0,0.0,False,0.0,False,0.0,periodistica,0.6,0.8
